# UnifyComm: intent model retraining (Plan B)

The original intent weights were lost, so this notebook retrains the model from the
documented recipe of the team's two Colab notebooks and the saved label mapping.

**How to run (about 10–15 minutes):**
1. `Runtime → Change runtime type → T4 GPU`, then `Save`.
2. `Runtime → Run all`.
3. When it finishes, the browser downloads `unifycomm_intent_repro-v1.zip`. Allow the download if asked.
4. The last cell also copies the zip to your Google Drive (`MyDrive/UnifyComm/models/intent/`). Approve the Drive prompt.
5. Put the zip in `C:\UnifyComm\models\intent\` and tell Claude.

**What it reproduces:** `all-MiniLM-L6-v2` encoder, mean pooling, dropout 0.1, linear head,
max length 64, the same `clean_text`, the same 213 unified labels and label order.
Stage 1 trains on Banking77 (77 labels); Stage 2 trains jointly on Banking77 + CLINC150 "plus"
(213 labels) with encoder LR 1e-5 and head LR 1e-4.

**What it fixes:** the original Stage 1 picked its best epoch on the *test* set. Here, model
selection uses validation data (10% of Banking77 train, plus CLINC150 validation) and each
test set is used once, at the end. The weights are saved as `model.safetensors` and copied to Drive.

In [ ]:
# Setup: detect Colab and install pinned libraries (Colab already has CUDA PyTorch).
import os
import subprocess
import sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content/sample_data")
DRY_RUN = os.environ.get("UC_DRY_RUN") == "1"  # local CPU smoke test with a tiny sample

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers==5.18.0", "datasets>=3.0", "safetensors", "scikit-learn"],
        check=True,
    )
print(f"Colab: {IN_COLAB} | dry run: {DRY_RUN}")

In [ ]:
import copy
import hashlib
import json
import random
import re
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import transformers
from datasets import load_dataset
from safetensors.torch import save_file
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and not DRY_RUN:
    raise RuntimeError(
        "No GPU found. In Colab: Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all."
    )
print(f"torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Configuration. Every external input is pinned to an exact revision.
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BASE_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
B77_ID, B77_REVISION = "legacy-datasets/banking77", "f54121560de48f2852f90be299010d1d6dc612ec"
CLINC_ID, CLINC_CONFIG = "clinc/clinc_oos", "plus"
CLINC_REVISION = "155b9c710419136e17307b80d0a13e68cd46b4ec"

MAX_LENGTH = 64
DROPOUT = 0.1
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 256
B77_VAL_FRACTION = 0.10
STAGE1 = {"epochs": 5, "lr": 2e-5, "warmup_steps": 100, "weight_decay": 0.01, "patience": 2}
STAGE2 = {"epochs": 5, "encoder_lr": 1e-5, "head_lr": 1e-4, "warmup_steps": 100,
          "weight_decay": 0.01, "patience": 2}
MODEL_VERSION_NAME = "repro-v1"

if DRY_RUN:  # tiny, fast settings for a local CPU check of the whole pipeline
    BATCH_SIZE, EVAL_BATCH_SIZE = 16, 64
    STAGE1 = {**STAGE1, "epochs": 1, "warmup_steps": 2}
    STAGE2 = {**STAGE2, "epochs": 1, "warmup_steps": 2}

OUT_DIR = Path(os.environ.get("UC_OUT_DIR", "/content/repro-v1" if IN_COLAB else "repro-v1"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT_DIR.resolve())

In [ ]:
# The team's unified label mapping (213 labels, exact order). Embedded so the notebook is self-contained.
MAPPING_JSON = r'''{"num_labels":213,"banking77_labels":["activate_my_card","age_limit","apple_pay_or_google_pay","atm_support","automatic_top_up","balance_not_updated_after_bank_transfer","balance_not_updated_after_cheque_or_cash_deposit","beneficiary_not_allowed","cancel_transfer","card_about_to_expire","card_acceptance","card_arrival","card_delivery_estimate","card_linking","card_not_working","card_payment_fee_charged","card_payment_not_recognised","card_payment_wrong_exchange_rate","card_swallowed","cash_withdrawal_charge","cash_withdrawal_not_recognised","change_pin","compromised_card","contactless_not_working","country_support","declined_card_payment","declined_cash_withdrawal","declined_transfer","direct_debit_payment_not_recognised","disposable_card_limits","edit_personal_details","exchange_charge","exchange_rate","exchange_via_app","extra_charge_on_statement","failed_transfer","fiat_currency_support","get_disposable_virtual_card","get_physical_card","getting_spare_card","getting_virtual_card","lost_or_stolen_card","lost_or_stolen_phone","order_physical_card","passcode_forgotten","pending_card_payment","pending_cash_withdrawal","pending_top_up","pending_transfer","pin_blocked","receiving_money","Refund_not_showing_up","request_refund","reverted_card_payment?","supported_cards_and_currencies","terminate_account","top_up_by_bank_transfer_charge","top_up_by_card_charge","top_up_by_cash_or_cheque","top_up_failed","top_up_limits","top_up_reverted","topping_up_by_card","transaction_charged_twice","transfer_fee_charged","transfer_into_account","transfer_not_received_by_recipient","transfer_timing","unable_to_verify_identity","verify_my_identity","verify_source_of_funds","verify_top_up","virtual_card_not_working","visa_or_mastercard","why_verify_identity","wrong_amount_of_cash_received","wrong_exchange_rate_for_cash_withdrawal"],"clinc150_added_labels":["restaurant_reviews","nutrition_info","account_blocked","oil_change_how","time","weather","redeem_rewards","interest_rate","gas_type","accept_reservations","smart_home","user_name","repeat","whisper_mode","what_are_your_hobbies","order","jump_start","schedule_meeting","meeting_schedule","freeze_account","what_song","meaning_of_life","restaurant_reservation","traffic","make_call","text","improve_credit_score","change_language","no","measurement_conversion","timer","flip_coin","do_you_have_pets","tell_joke","last_maintenance","uber","car_rental","credit_limit","oos","shopping_list","expiration_date","routing","meal_suggestion","tire_change","todo_list","rewards_balance","change_accent","vaccines","reminder_update","food_last","change_ai_name","who_do_you_work_for","share_location","international_visa","calendar","translate","carry_on","book_flight","insurance_change","todo_list_update","timezone","cancel_reservation","credit_score","report_fraud","directions","spelling","insurance","what_is_your_name","reminder","where_are_you_from","distance","payday","flight_status","find_phone","greeting","alarm","confirm_reservation","cook_time","damaged_card","reset_settings","new_card","roll_dice","income","taxes","date","who_made_you","pto_request","tire_pressure","how_old_are_you","rollover_401k","pto_request_status","how_busy","application_status","recipe","calendar_update","play_music","yes","credit_limit_change","gas","ingredients_list","lost_luggage","goodbye","what_can_i_ask_you","book_hotel","are_you_a_bot","next_song","change_speed","plug_type","maybe","w2","oil_change_when","thank_you","shopping_list_update","pto_balance","order_checks","travel_alert","fun_fact","sync_device","schedule_maintenance","apr","ingredient_substitution","calories","current_location","calculator","definition","next_holiday","update_playlist","mpg","min_payment","change_user_name","restaurant_suggestion","travel_notification","cancel","pto_used","travel_suggestion","change_volume"],"overlap_mapping":{"transfer":"balance_not_updated_after_bank_transfer","balance":"balance_not_updated_after_bank_transfer","pin_change":"pin_blocked","exchange_rate":"exchange_rate","direct_deposit":"beneficiary_not_allowed","pay_bill":"pending_top_up","spending_history":"transaction_charged_twice","bill_balance":"balance_not_updated_after_bank_transfer","bill_due":"pending_top_up","report_lost_card":"lost_or_stolen_card","replacement_card_duration":"card_delivery_estimate","card_declined":"declined_card_payment","international_fees":"exchange_charge","transactions":"transaction_charged_twice","order_status":"card_delivery_estimate","update_personal_information":"edit_personal_details"},"label2id":{"activate_my_card":0,"age_limit":1,"apple_pay_or_google_pay":2,"atm_support":3,"automatic_top_up":4,"balance_not_updated_after_bank_transfer":5,"balance_not_updated_after_cheque_or_cash_deposit":6,"beneficiary_not_allowed":7,"cancel_transfer":8,"card_about_to_expire":9,"card_acceptance":10,"card_arrival":11,"card_delivery_estimate":12,"card_linking":13,"card_not_working":14,"card_payment_fee_charged":15,"card_payment_not_recognised":16,"card_payment_wrong_exchange_rate":17,"card_swallowed":18,"cash_withdrawal_charge":19,"cash_withdrawal_not_recognised":20,"change_pin":21,"compromised_card":22,"contactless_not_working":23,"country_support":24,"declined_card_payment":25,"declined_cash_withdrawal":26,"declined_transfer":27,"direct_debit_payment_not_recognised":28,"disposable_card_limits":29,"edit_personal_details":30,"exchange_charge":31,"exchange_rate":32,"exchange_via_app":33,"extra_charge_on_statement":34,"failed_transfer":35,"fiat_currency_support":36,"get_disposable_virtual_card":37,"get_physical_card":38,"getting_spare_card":39,"getting_virtual_card":40,"lost_or_stolen_card":41,"lost_or_stolen_phone":42,"order_physical_card":43,"passcode_forgotten":44,"pending_card_payment":45,"pending_cash_withdrawal":46,"pending_top_up":47,"pending_transfer":48,"pin_blocked":49,"receiving_money":50,"Refund_not_showing_up":51,"request_refund":52,"reverted_card_payment?":53,"supported_cards_and_currencies":54,"terminate_account":55,"top_up_by_bank_transfer_charge":56,"top_up_by_card_charge":57,"top_up_by_cash_or_cheque":58,"top_up_failed":59,"top_up_limits":60,"top_up_reverted":61,"topping_up_by_card":62,"transaction_charged_twice":63,"transfer_fee_charged":64,"transfer_into_account":65,"transfer_not_received_by_recipient":66,"transfer_timing":67,"unable_to_verify_identity":68,"verify_my_identity":69,"verify_source_of_funds":70,"verify_top_up":71,"virtual_card_not_working":72,"visa_or_mastercard":73,"why_verify_identity":74,"wrong_amount_of_cash_received":75,"wrong_exchange_rate_for_cash_withdrawal":76,"restaurant_reviews":77,"nutrition_info":78,"account_blocked":79,"oil_change_how":80,"time":81,"weather":82,"redeem_rewards":83,"interest_rate":84,"gas_type":85,"accept_reservations":86,"smart_home":87,"user_name":88,"repeat":89,"whisper_mode":90,"what_are_your_hobbies":91,"order":92,"jump_start":93,"schedule_meeting":94,"meeting_schedule":95,"freeze_account":96,"what_song":97,"meaning_of_life":98,"restaurant_reservation":99,"traffic":100,"make_call":101,"text":102,"improve_credit_score":103,"change_language":104,"no":105,"measurement_conversion":106,"timer":107,"flip_coin":108,"do_you_have_pets":109,"tell_joke":110,"last_maintenance":111,"uber":112,"car_rental":113,"credit_limit":114,"oos":115,"shopping_list":116,"expiration_date":117,"routing":118,"meal_suggestion":119,"tire_change":120,"todo_list":121,"rewards_balance":122,"change_accent":123,"vaccines":124,"reminder_update":125,"food_last":126,"change_ai_name":127,"who_do_you_work_for":128,"share_location":129,"international_visa":130,"calendar":131,"translate":132,"carry_on":133,"book_flight":134,"insurance_change":135,"todo_list_update":136,"timezone":137,"cancel_reservation":138,"credit_score":139,"report_fraud":140,"directions":141,"spelling":142,"insurance":143,"what_is_your_name":144,"reminder":145,"where_are_you_from":146,"distance":147,"payday":148,"flight_status":149,"find_phone":150,"greeting":151,"alarm":152,"confirm_reservation":153,"cook_time":154,"damaged_card":155,"reset_settings":156,"new_card":157,"roll_dice":158,"income":159,"taxes":160,"date":161,"who_made_you":162,"pto_request":163,"tire_pressure":164,"how_old_are_you":165,"rollover_401k":166,"pto_request_status":167,"how_busy":168,"application_status":169,"recipe":170,"calendar_update":171,"play_music":172,"yes":173,"credit_limit_change":174,"gas":175,"ingredients_list":176,"lost_luggage":177,"goodbye":178,"what_can_i_ask_you":179,"book_hotel":180,"are_you_a_bot":181,"next_song":182,"change_speed":183,"plug_type":184,"maybe":185,"w2":186,"oil_change_when":187,"thank_you":188,"shopping_list_update":189,"pto_balance":190,"order_checks":191,"travel_alert":192,"fun_fact":193,"sync_device":194,"schedule_maintenance":195,"apr":196,"ingredient_substitution":197,"calories":198,"current_location":199,"calculator":200,"definition":201,"next_holiday":202,"update_playlist":203,"mpg":204,"min_payment":205,"change_user_name":206,"restaurant_suggestion":207,"travel_notification":208,"cancel":209,"pto_used":210,"travel_suggestion":211,"change_volume":212},"id2label":{"0":"activate_my_card","1":"age_limit","2":"apple_pay_or_google_pay","3":"atm_support","4":"automatic_top_up","5":"balance_not_updated_after_bank_transfer","6":"balance_not_updated_after_cheque_or_cash_deposit","7":"beneficiary_not_allowed","8":"cancel_transfer","9":"card_about_to_expire","10":"card_acceptance","11":"card_arrival","12":"card_delivery_estimate","13":"card_linking","14":"card_not_working","15":"card_payment_fee_charged","16":"card_payment_not_recognised","17":"card_payment_wrong_exchange_rate","18":"card_swallowed","19":"cash_withdrawal_charge","20":"cash_withdrawal_not_recognised","21":"change_pin","22":"compromised_card","23":"contactless_not_working","24":"country_support","25":"declined_card_payment","26":"declined_cash_withdrawal","27":"declined_transfer","28":"direct_debit_payment_not_recognised","29":"disposable_card_limits","30":"edit_personal_details","31":"exchange_charge","32":"exchange_rate","33":"exchange_via_app","34":"extra_charge_on_statement","35":"failed_transfer","36":"fiat_currency_support","37":"get_disposable_virtual_card","38":"get_physical_card","39":"getting_spare_card","40":"getting_virtual_card","41":"lost_or_stolen_card","42":"lost_or_stolen_phone","43":"order_physical_card","44":"passcode_forgotten","45":"pending_card_payment","46":"pending_cash_withdrawal","47":"pending_top_up","48":"pending_transfer","49":"pin_blocked","50":"receiving_money","51":"Refund_not_showing_up","52":"request_refund","53":"reverted_card_payment?","54":"supported_cards_and_currencies","55":"terminate_account","56":"top_up_by_bank_transfer_charge","57":"top_up_by_card_charge","58":"top_up_by_cash_or_cheque","59":"top_up_failed","60":"top_up_limits","61":"top_up_reverted","62":"topping_up_by_card","63":"transaction_charged_twice","64":"transfer_fee_charged","65":"transfer_into_account","66":"transfer_not_received_by_recipient","67":"transfer_timing","68":"unable_to_verify_identity","69":"verify_my_identity","70":"verify_source_of_funds","71":"verify_top_up","72":"virtual_card_not_working","73":"visa_or_mastercard","74":"why_verify_identity","75":"wrong_amount_of_cash_received","76":"wrong_exchange_rate_for_cash_withdrawal","77":"restaurant_reviews","78":"nutrition_info","79":"account_blocked","80":"oil_change_how","81":"time","82":"weather","83":"redeem_rewards","84":"interest_rate","85":"gas_type","86":"accept_reservations","87":"smart_home","88":"user_name","89":"repeat","90":"whisper_mode","91":"what_are_your_hobbies","92":"order","93":"jump_start","94":"schedule_meeting","95":"meeting_schedule","96":"freeze_account","97":"what_song","98":"meaning_of_life","99":"restaurant_reservation","100":"traffic","101":"make_call","102":"text","103":"improve_credit_score","104":"change_language","105":"no","106":"measurement_conversion","107":"timer","108":"flip_coin","109":"do_you_have_pets","110":"tell_joke","111":"last_maintenance","112":"uber","113":"car_rental","114":"credit_limit","115":"oos","116":"shopping_list","117":"expiration_date","118":"routing","119":"meal_suggestion","120":"tire_change","121":"todo_list","122":"rewards_balance","123":"change_accent","124":"vaccines","125":"reminder_update","126":"food_last","127":"change_ai_name","128":"who_do_you_work_for","129":"share_location","130":"international_visa","131":"calendar","132":"translate","133":"carry_on","134":"book_flight","135":"insurance_change","136":"todo_list_update","137":"timezone","138":"cancel_reservation","139":"credit_score","140":"report_fraud","141":"directions","142":"spelling","143":"insurance","144":"what_is_your_name","145":"reminder","146":"where_are_you_from","147":"distance","148":"payday","149":"flight_status","150":"find_phone","151":"greeting","152":"alarm","153":"confirm_reservation","154":"cook_time","155":"damaged_card","156":"reset_settings","157":"new_card","158":"roll_dice","159":"income","160":"taxes","161":"date","162":"who_made_you","163":"pto_request","164":"tire_pressure","165":"how_old_are_you","166":"rollover_401k","167":"pto_request_status","168":"how_busy","169":"application_status","170":"recipe","171":"calendar_update","172":"play_music","173":"yes","174":"credit_limit_change","175":"gas","176":"ingredients_list","177":"lost_luggage","178":"goodbye","179":"what_can_i_ask_you","180":"book_hotel","181":"are_you_a_bot","182":"next_song","183":"change_speed","184":"plug_type","185":"maybe","186":"w2","187":"oil_change_when","188":"thank_you","189":"shopping_list_update","190":"pto_balance","191":"order_checks","192":"travel_alert","193":"fun_fact","194":"sync_device","195":"schedule_maintenance","196":"apr","197":"ingredient_substitution","198":"calories","199":"current_location","200":"calculator","201":"definition","202":"next_holiday","203":"update_playlist","204":"mpg","205":"min_payment","206":"change_user_name","207":"restaurant_suggestion","208":"travel_notification","209":"cancel","210":"pto_used","211":"travel_suggestion","212":"change_volume"}}'''
if MAPPING_JSON is None:
    MAPPING_JSON = Path(os.environ["UC_MAPPING_PATH"]).read_text(encoding="utf-8")
MAPPING = json.loads(MAPPING_JSON)

ID2LABEL = {int(k): v for k, v in MAPPING["id2label"].items()}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
BANKING_LABELS = MAPPING["banking77_labels"]
OVERLAP = MAPPING["overlap_mapping"]
NUM_LABELS = len(ID2LABEL)
assert NUM_LABELS == 213 and [ID2LABEL[i] for i in range(77)] == BANKING_LABELS
print(f"{NUM_LABELS} unified labels, {len(OVERLAP)} CLINC→Banking77 merges")

In [ ]:
# Text cleaning: identical to the team's notebook. The app must apply the same function at inference.
def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\'\-]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


assert clean_text("my card  was   stolen #help @bank") == "my card was stolen help bank"

In [ ]:
# Load data and map everything to unified label IDs.
b77 = load_dataset(B77_ID, revision=B77_REVISION)
clinc = load_dataset(CLINC_ID, CLINC_CONFIG, revision=CLINC_REVISION)

b77_names = b77["train"].features["label"].names
assert b77_names == BANKING_LABELS, "Banking77 label order differs from the unified mapping"
clinc_names = clinc["train"].features["intent"].names
clinc_to_unified = [LABEL2ID[OVERLAP.get(name, name)] for name in clinc_names]


def rows(split, text_col, label_col, mapper=None):
    texts = list(split[text_col])
    labels = list(split[label_col])
    if mapper is not None:
        labels = [mapper[x] for x in labels]
    return texts, labels


b77_train_texts, b77_train_labels = rows(b77["train"], "text", "label")
b77_test = rows(b77["test"], "text", "label")
clinc_train = rows(clinc["train"], "text", "intent", clinc_to_unified)
clinc_val = rows(clinc["validation"], "text", "intent", clinc_to_unified)
clinc_test = rows(clinc["test"], "text", "intent", clinc_to_unified)

idx_tr, idx_val = train_test_split(
    list(range(len(b77_train_texts))), test_size=B77_VAL_FRACTION,
    stratify=b77_train_labels, random_state=SEED,
)
b77_train = ([b77_train_texts[i] for i in idx_tr], [b77_train_labels[i] for i in idx_tr])
b77_val = ([b77_train_texts[i] for i in idx_val], [b77_train_labels[i] for i in idx_val])

if DRY_RUN:
    def head(pair, n):
        return pair[0][:n], pair[1][:n]
    b77_train, b77_val, b77_test = head(b77_train, 256), head(b77_val, 64), head(b77_test, 64)
    clinc_train, clinc_val, clinc_test = head(clinc_train, 256), head(clinc_val, 64), head(clinc_test, 64)

for name, pair in [("banking77 train", b77_train), ("banking77 val", b77_val),
                   ("banking77 test", b77_test), ("clinc150 train", clinc_train),
                   ("clinc150 val", clinc_val), ("clinc150 test", clinc_test)]:
    print(f"{name:16s} {len(pair[0]):>6,}")

In [ ]:
# Tokenize once into tensors.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)


def to_dataset(pair) -> TensorDataset:
    enc = tokenizer([clean_text(t) for t in pair[0]], padding="max_length", truncation=True,
                    max_length=MAX_LENGTH, return_tensors="pt")
    return TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(pair[1]))


def concat(a, b):
    return a[0] + b[0], a[1] + b[1]


ds = {
    "s1_train": to_dataset(b77_train),
    "s1_val": to_dataset(b77_val),
    "s2_train": to_dataset(concat(b77_train, clinc_train)),
    "s2_val": to_dataset(concat(b77_val, clinc_val)),
    "b77_test": to_dataset(b77_test),
    "clinc_test": to_dataset(clinc_test),
}
print({k: len(v) for k, v in ds.items()})

In [ ]:
# Model: same architecture as the team's IntentClassifier (state-dict keys encoder.* and classifier.*).
class IntentClassifier(nn.Module):
    def __init__(self, encoder: nn.Module, num_labels: int, dropout_rate: float = DROPOUT):
        super().__init__()
        self.encoder = encoder
        self.hidden_size = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    @staticmethod
    def mean_pooling(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        return torch.sum(last_hidden_state * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pooling(out.last_hidden_state, attention_mask)
        return self.classifier(self.dropout(pooled))


def param_groups(model: IntentClassifier, encoder_lr: float, head_lr: float, weight_decay: float):
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    groups = []
    for module, lr in ((model.encoder, encoder_lr), (model.classifier, head_lr)):
        named = list(module.named_parameters())
        groups.append({"params": [p for n, p in named if not n.endswith(no_decay)],
                       "lr": lr, "weight_decay": weight_decay})
        groups.append({"params": [p for n, p in named if n.endswith(no_decay)],
                       "lr": lr, "weight_decay": 0.0})
    return groups

In [ ]:
# Training and evaluation helpers (plain PyTorch loop, fp16 autocast on GPU, gradient clipping 1.0).
USE_AMP = DEVICE.type == "cuda"


@torch.no_grad()
def predict(model, dataset):
    model.eval()
    all_logits = []
    for ids, mask, _ in DataLoader(dataset, batch_size=EVAL_BATCH_SIZE):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(ids.to(DEVICE), mask.to(DEVICE))
        all_logits.append(logits.float().cpu())
    logits = torch.cat(all_logits)
    return logits.argmax(-1).numpy(), dataset.tensors[2].numpy()


def scores(y_pred, y_true) -> dict:
    present = sorted(set(y_true.tolist()))
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_f1_true_labels": float(f1_score(y_true, y_pred, labels=present, average="macro",
                                               zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "n": int(len(y_true)),
    }


def train_stage(model, train_ds, val_ds, groups, epochs, warmup_steps, patience, name):
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    optimizer = torch.optim.AdamW(groups)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, len(loader) * epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    loss_fn = nn.CrossEntropyLoss()
    best_acc, best_state, bad_epochs, history = -1.0, None, 0, []
    for epoch in range(1, epochs + 1):
        model.train()
        started, losses = time.time(), []
        for ids, mask, labels in loader:
            ids, mask, labels = ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(ids, mask)
            loss = loss_fn(logits.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(loss.item())
        val = scores(*predict(model, val_ds))
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "seconds": round(time.time() - started, 1),
               **{f"val_{k}": v for k, v in val.items() if k != "n"}}
        history.append(row)
        print(f"[{name}] epoch {epoch}: loss {row['train_loss']:.4f} | val acc {val['accuracy']:.4f} "
              f"| val macro-F1 {val['macro_f1']:.4f} | {row['seconds']}s")
        if val["accuracy"] > best_acc:
            best_acc, bad_epochs = val["accuracy"], 0
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"[{name}] early stop")
                break
    model.load_state_dict(best_state)
    best_epoch = max(history, key=lambda r: r["val_accuracy"])["epoch"]
    print(f"[{name}] best epoch {best_epoch}, val acc {best_acc:.4f}")
    return history, best_epoch

In [ ]:
# Stage 1: Banking77 only (77 labels).
seed_all(SEED)
encoder = AutoModel.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
stage1 = IntentClassifier(encoder, num_labels=77).to(DEVICE)
t0 = time.time()
s1_history, s1_best = train_stage(
    stage1, ds["s1_train"], ds["s1_val"],
    param_groups(stage1, STAGE1["lr"], STAGE1["lr"], STAGE1["weight_decay"]),
    STAGE1["epochs"], STAGE1["warmup_steps"], STAGE1["patience"], "stage1",
)
s1_test = scores(*predict(stage1, ds["b77_test"]))
print(f"Stage 1 Banking77 test: {s1_test}  ({time.time() - t0:.0f}s)")

In [ ]:
# Stage 2: joint Banking77 + CLINC150 (213 labels). Head rows 0..76 start from the Stage 1 head.
seed_all(SEED)
stage2 = IntentClassifier(stage1.encoder, num_labels=NUM_LABELS).to(DEVICE)
with torch.no_grad():
    stage2.classifier.weight[:77].copy_(stage1.classifier.weight)
    stage2.classifier.bias[:77].copy_(stage1.classifier.bias)
del stage1
t0 = time.time()
s2_history, s2_best = train_stage(
    stage2, ds["s2_train"], ds["s2_val"],
    param_groups(stage2, STAGE2["encoder_lr"], STAGE2["head_lr"], STAGE2["weight_decay"]),
    STAGE2["epochs"], STAGE2["warmup_steps"], STAGE2["patience"], "stage2",
)
print(f"Stage 2 training time: {time.time() - t0:.0f}s")

In [ ]:
# Final test evaluation: each test set is used exactly once, here.
b77_pred, b77_true = predict(stage2, ds["b77_test"])
clinc_pred, clinc_true = predict(stage2, ds["clinc_test"])
b77_scores, clinc_scores = scores(b77_pred, b77_true), scores(clinc_pred, clinc_true)
oos_id = LABEL2ID["oos"]
oos_mask = clinc_true == oos_id
oos_recall = float((clinc_pred[oos_mask] == oos_id).mean()) if oos_mask.any() else None
print("Banking77 test :", b77_scores)
print("CLINC150 test  :", clinc_scores)
print("CLINC150 out-of-scope recall:", oos_recall)


def report(y_true, y_pred):
    present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    return classification_report(y_true, y_pred, labels=present,
                                 target_names=[ID2LABEL[i] for i in present],
                                 output_dict=True, zero_division=0)

In [ ]:
# Golden check: the six example predictions shown in the team's slides.
GOLDEN = [
    ("I can't access my account anymore", "account_blocked", 0.5358),
    ("What's the exchange rate for USD to EUR?", "exchange_rate", 0.9976),
    ("My card was stolen yesterday", "lost_or_stolen_card", 0.9700),
    ("Can you help me book a flight to London?", "book_flight", 0.9646),
    ("I was charged twice for the same transaction", "transaction_charged_twice", 0.9999),
    ("What's the weather like today?", "weather", 0.9910),
]


@torch.no_grad()
def top_k(model, text, k=3):
    model.eval()
    enc = tokenizer([clean_text(text)], padding="max_length", truncation=True,
                    max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
    probs = torch.softmax(model(enc["input_ids"], enc["attention_mask"]).float(), -1)[0].cpu()
    values, indices = probs.topk(k)
    return [(ID2LABEL[int(i)], round(float(v), 4)) for v, i in zip(values, indices)]


golden_results = []
for text, expected, original_conf in GOLDEN:
    preds = top_k(stage2, text)
    ok = preds[0][0] == expected
    golden_results.append({"text": text, "expected": expected, "original_confidence": original_conf,
                           "top3": preds, "match": ok})
    print(f"{'OK ' if ok else 'DIFF'} {text!r}\n     -> {preds}   (original: {expected} {original_conf})")
print(f"Golden matches: {sum(r['match'] for r in golden_results)}/{len(GOLDEN)}")

In [ ]:
# Export: safetensors weights + config + tokenizer + labels + metrics, then zip.
state = {k: v.detach().cpu().contiguous() for k, v in stage2.state_dict().items()}
save_file(state, str(OUT_DIR / "model.safetensors"),
          metadata={"format": "pt", "architecture": "IntentClassifier"})
stage2.encoder.config.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
weights_sha256 = hashlib.sha256((OUT_DIR / "model.safetensors").read_bytes()).hexdigest()


def write_json(name, obj):
    (OUT_DIR / name).write_text(json.dumps(obj, indent=2), encoding="utf-8")


write_json("id2label.json", {str(i): ID2LABEL[i] for i in range(NUM_LABELS)})
write_json("label2id.json", LABEL2ID)
write_json("unified_label_mapping.json", MAPPING)
write_json("model_meta.json", {
    "model_name": BASE_MODEL, "base_revision": BASE_REVISION, "architecture": "IntentClassifier",
    "pooling": "mean", "num_labels": NUM_LABELS, "hidden_size": stage2.hidden_size,
    "dropout_rate": DROPOUT, "max_length": MAX_LENGTH, "text_cleaning": "clean_text_v1",
    "state_dict_prefixes": ["encoder.", "classifier."], "version_name": MODEL_VERSION_NAME,
    "weights_sha256": weights_sha256,
})
write_json("final_metrics.json", {
    "version_name": MODEL_VERSION_NAME, "dry_run": DRY_RUN,
    "banking77_test": b77_scores, "clinc150_test": clinc_scores,
    "clinc150_oos_recall": oos_recall,
    "banking77_test_stage1_77way": s1_test,
    "regression_delta_banking77": b77_scores["accuracy"] - s1_test["accuracy"],
    "stage1_best_epoch": s1_best, "stage2_best_epoch": s2_best,
    "golden_matches": f"{sum(r['match'] for r in golden_results)}/{len(GOLDEN)}",
})
write_json("training_history.json", {"stage1": s1_history, "stage2": s2_history,
                                     "config": {"stage1": STAGE1, "stage2": STAGE2,
                                                "batch_size": BATCH_SIZE, "seed": SEED,
                                                "b77_val_fraction": B77_VAL_FRACTION}})
write_json("classification_report_banking77.json", report(b77_true, b77_pred))
write_json("classification_report_clinc150.json", report(clinc_true, clinc_pred))
write_json("golden_results.json", golden_results)
write_json("environment.json", {
    "torch": torch.__version__, "transformers": transformers.__version__,
    "device": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
    "datasets": {"banking77": [B77_ID, B77_REVISION],
                 "clinc150": [CLINC_ID, CLINC_CONFIG, CLINC_REVISION]},
})

ZIP_PATH = OUT_DIR.parent / f"unifycomm_intent_{MODEL_VERSION_NAME}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.iterdir()):
        zf.write(f, arcname=f"{MODEL_VERSION_NAME}/{f.name}")
print(f"Saved {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB), weights sha256 {weights_sha256[:12]}")

In [ ]:
# Download the zip to your computer.
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))

In [ ]:
# Backup copy in Google Drive so the weights cannot be lost again.
SAVE_TO_DRIVE = True
if IN_COLAB and SAVE_TO_DRIVE:
    import shutil

    from google.colab import drive
    drive.mount("/content/drive")
    target = Path("/content/drive/MyDrive/UnifyComm/models/intent")
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_PATH, target / ZIP_PATH.name)
    print("Copied to", target / ZIP_PATH.name)